# 02 — Tune Neural Network

Compare pure LSTM, GRU, and TCN models that predict match win probability
from a single player's recent match history (no opponent data).
Best model logged to MLflow.

In [ ]:
input_dir = "data/processed"
n_trials = 50
max_epochs = 100
patience = 10
batch_size = 128
seq_len = 10

In [ ]:
import json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import lightning as L
import optuna
import mlflow
from torch.utils.data import Dataset, DataLoader
from optuna_integration.pytorch_lightning import PyTorchLightningPruningCallback

In [ ]:
# ── Load data ──
sequences = pd.read_parquet(f"{input_dir}/sequences.parquet")
with open(f"{input_dir}/seq_feature_cols.json") as f:
    seq_feat_cols = json.load(f)

info_train = pd.read_parquet(f"{input_dir}/info_train.parquet")
info_test = pd.read_parquet(f"{input_dir}/info_test.parquet")
y_train = pd.read_parquet(f"{input_dir}/y_train.parquet")["y"]

print(f"Sequences: {len(sequences)} rows, feat_dim: {len(seq_feat_cols)}")

In [ ]:
# ── Build per-player index into the sequences table ──
player_mask = {}
for player in pd.unique(sequences["player_id"]):
    mask = sequences["player_id"] == player
    player_mask[player] = np.where(mask.values)[0]

seq_array = sequences[seq_feat_cols].values.astype(np.float32)
seq_dates = sequences["match_date"].values
feat_dim = len(seq_feat_cols)
print(f"Players indexed: {len(player_mask)}")

In [ ]:
# ── Dataset ──


class MatchDataset(Dataset):
    """Single-player (seq, label) pairs."""

    def __init__(self, matches, seq_array, seq_dates, player_mask, seq_len, feat_dim):
        self.seq_array = seq_array
        self.seq_dates = seq_dates
        self.player_mask = player_mask
        self.seq_len = seq_len
        self.feat_dim = feat_dim
        self.matches = matches

    def __len__(self):
        return len(self.matches)

    def __getitem__(self, idx):
        row = self.matches.iloc[idx]
        match_date = row["match_date"]
        pm = self.player_mask.get(row["player_id"], np.array([], dtype=int))

        if len(pm) == 0:
            seq = np.zeros((self.seq_len, self.feat_dim), dtype=np.float32)
        else:
            idx_before = np.where(self.seq_dates[pm] < match_date)[0]
            if len(idx_before) == 0:
                seq = np.zeros((self.seq_len, self.feat_dim), dtype=np.float32)
            else:
                recent = pm[idx_before[-self.seq_len :]]
                seq = self.seq_array[recent]
                if len(seq) < self.seq_len:
                    pad = np.zeros((self.seq_len - len(seq), self.feat_dim), dtype=np.float32)
                    seq = np.vstack([pad, seq])

        label = float(row.get("match_won", row.get("y", 0)))
        return torch.from_numpy(seq), torch.tensor(label, dtype=torch.float32)


class MatchDataModule(L.LightningDataModule):
    def __init__(
        self,
        info_train,
        info_test,
        y_train,
        seq_array,
        seq_dates,
        player_mask,
        seq_len,
        feat_dim,
        batch_size,
    ):
        super().__init__()
        self.info_train = info_train
        self.info_test = info_test
        self.y_train = y_train
        self.seq_array = seq_array
        self.seq_dates = seq_dates
        self.player_mask = player_mask
        self.seq_len = seq_len
        self.feat_dim = feat_dim
        self.batch_size = batch_size

    def setup(self, _stage=None):
        train = self.info_train.copy()
        train["match_won"] = self.y_train
        train = train[train["player_id"].isin(self.player_mask)]
        test = self.info_test.copy()
        test = test[test["player_id"].isin(self.player_mask)]

        self.train_ds = MatchDataset(
            train,
            self.seq_array,
            self.seq_dates,
            self.player_mask,
            self.seq_len,
            self.feat_dim,
        )
        self.val_ds = MatchDataset(
            test,
            self.seq_array,
            self.seq_dates,
            self.player_mask,
            self.seq_len,
            self.feat_dim,
        )

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.batch_size,
            shuffle=True,
            num_workers=0,
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.batch_size * 2,
            shuffle=False,
            num_workers=0,
        )

In [ ]:
# ── Model ──


class TCNEncoder(nn.Module):
    def __init__(self, feat_dim, hidden_dim, num_layers=3, kernel_size=3, dropout=0.0):
        super().__init__()
        layers = []
        in_c = feat_dim
        for i in range(num_layers):
            dilation = 2**i
            layers.extend(
                [
                    nn.Conv1d(in_c, hidden_dim, kernel_size, padding="same", dilation=dilation),
                    nn.ReLU(),
                    nn.Dropout(dropout),
                ]
            )
            in_c = hidden_dim
        self.net = nn.Sequential(*layers)

    def forward(self, seq):
        x = self.net(seq.transpose(1, 2))
        return x.mean(dim=-1)


class PlayerPredictor(L.LightningModule):
    def __init__(
        self, feat_dim, hidden_dim=64, num_layers=1, dropout=0.0, encoder_type="lstm", _lr=1e-3
    ):
        super().__init__()
        self.save_hyperparameters()

        if encoder_type == "lstm":
            self.encoder = nn.LSTM(
                feat_dim,
                hidden_dim,
                num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0,
            )
        elif encoder_type == "gru":
            self.encoder = nn.GRU(
                feat_dim,
                hidden_dim,
                num_layers,
                batch_first=True,
                dropout=dropout if num_layers > 1 else 0,
            )
        elif encoder_type == "tcn":
            self.encoder = TCNEncoder(feat_dim, hidden_dim, num_layers, dropout=dropout)
        else:
            raise ValueError(f"Unknown encoder: {encoder_type}")

        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, 1),
        )

    def forward(self, seq):
        if isinstance(self.encoder, TCNEncoder):
            out = self.encoder(seq)
        else:
            result = self.encoder(seq)
            h_n = result[1] if not isinstance(result[1], tuple) else result[1][0]
            out = h_n[-1]
        return self.head(out).squeeze(-1)

    def training_step(self, batch, _batch_idx):
        seq, labels = batch
        logits = self(seq)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, labels)
        self.log("train_loss", loss, prog_bar=True)
        return loss

    def validation_step(self, batch, _batch_idx):
        seq, labels = batch
        logits = self(seq)
        loss = nn.functional.binary_cross_entropy_with_logits(logits, labels)
        probs = torch.sigmoid(logits)
        self.log("val_loss", loss, prog_bar=True, sync_dist=True)
        return {"probs": probs, "labels": labels}

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr)

In [ ]:
# ── Optuna objective ──


def objective(trial):
    encoder_type = trial.suggest_categorical("encoder", ["lstm", "gru", "tcn"])
    hidden_dim = trial.suggest_int("hidden_dim", 32, 256, log=True)
    num_layers = trial.suggest_int("num_layers", 1, 3)
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    model = PlayerPredictor(
        feat_dim=feat_dim,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        dropout=dropout,
        encoder_type=encoder_type,
        lr=lr,
    )

    datamodule = MatchDataModule(
        info_train,
        info_test,
        y_train,
        seq_array,
        seq_dates,
        player_mask,
        seq_len,
        feat_dim,
        batch_size,
    )

    trainer = L.Trainer(
        max_epochs=max_epochs,
        accelerator="auto",
        enable_progress_bar=False,
        enable_model_summary=False,
        callbacks=[
            L.pytorch.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=patience,
                mode="min",
            ),
            PyTorchLightningPruningCallback(trial, monitor="val_loss"),
        ],
        logger=False,
    )

    trainer.fit(model, datamodule=datamodule)
    return trainer.callback_metrics["val_loss"].item()


study = optuna.create_study(
    direction="minimize",
    pruner=optuna.pruners.MedianPruner(),
)
study.optimize(objective, n_trials=n_trials)
print(f"Best trial: value={study.best_trial.value:.4f}, params={study.best_trial.params}")

In [ ]:
# ── Log best to MLflow ──
mlflow.set_experiment("nn_models")
with mlflow.start_run():
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metric("best_val_loss", study.best_trial.value)
    mlflow.log_metric("n_trials", n_trials)
    print(f"Best NN params: {study.best_trial.params}")
    print(f"Logged to MLflow run: {mlflow.active_run().info.run_id}")